<a href="https://colab.research.google.com/github/aaronteork/CS5284_GML/blob/main/interactive_plot_draft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Imports
import sys
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from functools import lru_cache

# Colab widget support
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
# Mount drive and folders
drive.mount('/content/drive', force_remount=True)

FOLDER_NAME = 'cs5284_project'
PROJECT_DIR = Path(f'/content/drive/MyDrive/{FOLDER_NAME}')
DATA_DIR = PROJECT_DIR / "data"
SCRIPT_DIR = PROJECT_DIR / "scripts"

LOCAL_DATA_DIR = Path('/content/data')
LOCAL_DATA_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(SCRIPT_DIR.resolve()))

Mounted at /content/drive


In [ ]:
# Read data
import utils

data = utils.read_data(DATA_DIR, LOCAL_DATA_DIR)

Found 2 zip file(s): ['DE.zip', 'MD.zip']
Extracting DE.zip to /content/data...
Finished extracting DE.zip!
Extracting MD.zip to /content/data...
Finished extracting MD.zip!
DE: Symmetric = False


/usr/local/lib/python3.13/dist-packages/torch/_utils.py:358: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  result = torch.sparse_coo_tensor(


MD: Symmetric = False


In [ ]:
# Prepare road coordinates and accident records
map_data = {}

for state, info in data.items():
    node_file = next(
        (info["filepath"] / "Nodes").glob("node_features_*.csv")
    )
    nodes = pd.read_csv(node_file)

    adj = info["adj"].coalesce()
    src, dst = adj.indices().cpu().numpy()

    # Correct swapped latitude/longitude columns
    lat = nodes["lon"].to_numpy()
    lon = nodes["lat"].to_numpy()

    # Verify geographic coordinates
    assert np.all((-90 <= lat) & (lat <= 90))
    assert np.all((-180 <= lon) & (lon <= 180))
    assert len(nodes) == adj.shape[0]

    # One row per directed road
    roads = pd.DataFrame({
        "src": src,
        "dst": dst,
        "lat1": lat[src],
        "lon1": lon[src],
        "lat2": lat[dst],
        "lon2": lon[dst],
    })

    accidents = info["accidents"].copy()

    # Aggregate duplicate road-month records
    accidents = (
        accidents.groupby(
            ["year", "month", "node_1_idx", "node_2_idx"],
            as_index=False
        )["acc_count"].sum()
    )

    map_data[state] = {
        "roads": roads,
        "accidents": accidents,
        "center": {
            "lat": float(np.mean(lat)),
            "lon": float(np.mean(lon)),
        },
    }

    print(f"{state}: {len(roads):,} roads prepared")

DE: 116,196 roads prepared
MD: 580,526 roads prepared


In [ ]:
# Colors from low to high accident counts
RISK_COLORS = [
    "#FDE68A",  # Yellow
    "#FBBF24",
    "#FB923C",
    "#F97316",
    "#EF4444",
    "#991B1B",  # Dark red
]

MAX_ROADS = 3000
rng = np.random.default_rng(42)


def plot_accidents(state, year, month, aggregation):
    info = map_data[state]
    roads = info["roads"]
    accidents = info["accidents"]

    # Filter accident records
    selected = accidents[accidents["year"] == year]

    if aggregation == "Monthly":
        selected = selected[selected["month"] == month]

    # Sum accident counts by directed road
    counts = (
        selected.groupby(["node_1_idx", "node_2_idx"])["acc_count"]
        .sum()
        .rename("accidents")
    )

    df = roads.join(
        counts,
        on=["src", "dst"]
    )

    df["accidents"] = df["accidents"].fillna(0)

    # Select roads to display:
    # prioritize roads with accidents, plus random background roads
    positive = df[df["accidents"] > 0]
    zero = df[df["accidents"] == 0]

    if len(positive) >= MAX_ROADS:
        positive = positive.nlargest(MAX_ROADS, "accidents")
        zero = zero.iloc[:0]
    else:
        zero = zero.sample(
            n=min(MAX_ROADS - len(positive), len(zero)),
            random_state=42
        )

    sampled = pd.concat([zero, positive])

    fig = go.Figure()

    # Background: roads without accidents
    background = sampled[sampled["accidents"] == 0]

    if not background.empty:
        lats = np.column_stack([
            background["lat1"],
            background["lat2"],
            np.full(len(background), np.nan)
        ]).ravel()

        lons = np.column_stack([
            background["lon1"],
            background["lon2"],
            np.full(len(background), np.nan)
        ]).ravel()

        fig.add_trace(go.Scattermap(
            lat=lats,
            lon=lons,
            mode="lines",
            line=dict(color="#CBD5E1", width=1),
            name="No accidents",
            hoverinfo="skip"
        ))

    # Positive accident counts
    positive = sampled[sampled["accidents"] > 0].copy()

    if not positive.empty:
        # Logarithmic color scaling for skewed counts
        log_counts = np.log1p(positive["accidents"])

        max_log = np.log1p(
            max(1, df["accidents"].max())
        )

        positive["color_bin"] = np.clip(
            (log_counts / max_log * len(RISK_COLORS)).astype(int),
            0,
            len(RISK_COLORS) - 1
        )

        for bin_id, color in enumerate(RISK_COLORS):
            group = positive[positive["color_bin"] == bin_id]

            if group.empty:
                continue

            lats = np.column_stack([
                group["lat1"],
                group["lat2"],
                np.full(len(group), np.nan)
            ]).ravel()

            lons = np.column_stack([
                group["lon1"],
                group["lon2"],
                np.full(len(group), np.nan)
            ]).ravel()

            fig.add_trace(go.Scattermap(
                lat=lats,
                lon=lons,
                mode="lines",
                line=dict(color=color, width=3),
                hoverinfo="skip",
                showlegend=False
            ))

        # Invisible markers for road-level hover details
        fig.add_trace(go.Scattermap(
            lat=(positive["lat1"] + positive["lat2"]) / 2,
            lon=(positive["lon1"] + positive["lon2"]) / 2,
            mode="markers",
            marker=dict(size=8, opacity=0.01),
            customdata=positive[
                ["src", "dst", "accidents"]
            ].to_numpy(),
            hovertemplate=(
                "<b>Road segment</b>"
                "<br>Source: %{customdata[0]}"
                "<br>Target: %{customdata[1]}"
                "<br>Accidents: %{customdata[2]:.0f}"
                "<extra></extra>"
            ),
            showlegend=False
        ))

    period = (
        str(year)
        if aggregation == "Yearly"
        else f"{year}-{month:02d}"
    )

    fig.update_layout(
        title=dict(
            text=f"Traffic Accidents in {state} | {period}",
            font=dict(size=20)
        ),
        map=dict(
            style="carto-positron",
            center=info["center"],
            zoom=8 if state == "DE" else 7
        ),
        height=680,
        margin=dict(l=0, r=0, t=65, b=0),
        showlegend=False,
        paper_bgcolor="white"
    )

    fig.show()

    print(
        f"Period: {period} | "
        f"Total accidents: {df['accidents'].sum():,.0f} | "
        f"Roads with accidents: {(df['accidents'] > 0).sum():,} | "
        f"Roads displayed: {len(sampled):,}"
    )

In [ ]:
# Sidebar controls
state_widget = widgets.Dropdown(
    options=[("Delaware", "DE"), ("Maryland", "MD")],
    value="DE",
    description="State:",
    style={"description_width": "75px"}
)

aggregation_widget = widgets.ToggleButtons(
    options=["Monthly", "Yearly"],
    value="Monthly",
    description="View:",
    style={"description_width": "75px"}
)

year_widget = widgets.Dropdown(
    description="Year:",
    style={"description_width": "75px"}
)

month_widget = widgets.Dropdown(
    options=[
        ("January", 1), ("February", 2), ("March", 3),
        ("April", 4), ("May", 5), ("June", 6),
        ("July", 7), ("August", 8), ("September", 9),
        ("October", 10), ("November", 11), ("December", 12)
    ],
    description="Month:",
    style={"description_width": "75px"}
)

update_button = widgets.Button(
    description="Update Map",
    button_style="primary",
    icon="refresh",
    layout=widgets.Layout(width="180px")
)

map_output = widgets.Output()


def update_dates(*args):
    accidents = map_data[state_widget.value]["accidents"]

    years = sorted(accidents["year"].unique().tolist())
    current_year = year_widget.value

    year_widget.options = years
    year_widget.value = (
        current_year if current_year in years else years[-1]
    )

    available_months = sorted(
        accidents.loc[
            accidents["year"] == year_widget.value,
            "month"
        ].unique().tolist()
    )

    current_month = month_widget.value

    month_widget.options = [
        (pd.Timestamp(2000, m, 1).strftime("%B"), m)
        for m in available_months
    ]

    month_widget.value = (
        current_month
        if current_month in available_months
        else available_months[-1]
    )


def toggle_aggregation(change):
    month_widget.disabled = (
        aggregation_widget.value == "Yearly"
    )


def update_map(button=None):
    with map_output:
        map_output.clear_output(wait=True)

        plot_accidents(
            state=state_widget.value,
            year=year_widget.value,
            month=month_widget.value,
            aggregation=aggregation_widget.value
        )


state_widget.observe(update_dates, names="value")
year_widget.observe(update_dates, names="value")
aggregation_widget.observe(toggle_aggregation, names="value")
update_button.on_click(update_map)

update_dates()

# Dashboard layout
sidebar = widgets.VBox(
    [
        widgets.HTML("<h3>Map Filters</h3>"),
        state_widget,
        aggregation_widget,
        year_widget,
        month_widget,
        update_button,
    ],
    layout=widgets.Layout(
        width="230px",
        padding="15px",
        border="1px solid #dddddd"
    )
)

dashboard = widgets.HBox(
    [sidebar, map_output],
    layout=widgets.Layout(
        align_items="flex-start"
    )
)

display(
    widgets.HTML(
        "<h2>Traffic Accident Risk Explorer</h2>"
        "<p>Explore monthly and yearly road-level accidents.</p>"
    ),
    dashboard
)

update_map()

HTML(value='<h2>Traffic Accident Risk Explorer</h2><p>Explore monthly and yearly road-level accidents.</p>')